In [10]:
import numpy as np

latent_vectors = np.load(
    "../diffusion/amp_latent_vectors.npy"
)

print(latent_vectors.shape)

(8400, 128)


In [11]:
import pandas as pd

amp_df = pd.read_csv(
    "../data/processed_data&code/clean_amp_dataset.csv"
)

print(amp_df.shape)

(8400, 6)


In [12]:
lengths = amp_df["Sequence"].str.len()

print(lengths.describe())
print("Max length:", lengths.max())

count    8400.000000
mean       23.339405
std        12.351431
min         5.000000
25%        14.000000
50%        20.000000
75%        30.000000
max        60.000000
Name: Sequence, dtype: float64
Max length: 60


ESM embeddings were never trained to be decoded back into sequences, so it may reconstruct poorly

In [13]:
#amino-acid vocabulary dictionary
AA_VOCAB = "ACDEFGHIKLMNPQRSTVWY"

aa_to_idx = {
    aa: i + 1
    for i, aa in enumerate(AA_VOCAB)
}

aa_to_idx["PAD"] = 0

idx_to_aa = {
    v: k
    for k, v in aa_to_idx.items()
}

print(aa_to_idx)

{'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15, 'S': 16, 'T': 17, 'V': 18, 'W': 19, 'Y': 20, 'PAD': 0}


In [14]:
#encode sequences so that decoder can learn
MAX_LEN = 60

encoded_sequences = []

for seq in amp_df["Sequence"]:

    encoded = [
        aa_to_idx[aa]
        for aa in seq
    ]

    encoded += [0] * (
        MAX_LEN - len(encoded)
    )

    encoded_sequences.append(encoded)

import numpy as np

encoded_sequences = np.array(
    encoded_sequences
)

print(encoded_sequences.shape)

(8400, 60)


In [15]:
print(encoded_sequences.shape)
print(encoded_sequences[:2])

(8400, 60)
[[16 10  6 13  1  8  9  1 17 15 14 18  2 13  9  1 17 15  5 18 17 18 16  2
   9  9 16  3  2 14  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0]
 [18 17 16 19 16 10  2 17 13  6  2 17 16 13  6  6  6 16 12  2 16  5  2  2
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0]]


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    latent_vectors,
    encoded_sequences,
    test_size=0.1,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(7560, 128)
(840, 128)


In [20]:
#Converting the Numpy arrays into Pytorch tensors
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

print(type(X_train_tensor))
print(type(y_train_tensor))

<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [21]:
import torch.nn as nn

class SequenceDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1260)

        )

    def forward(self, x):

        out = self.net(x)

        out = out.view(
            -1,
            60,
            21
        )

        return out


decoder = SequenceDecoder()

print(decoder)

SequenceDecoder(
  (net): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1260, bias=True)
  )
)


In [22]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

decoder = decoder.to(device)

print(device)

cuda


In [23]:
#Craeting datasets using the new tensor variables
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

print(len(train_dataset))

7560


In [24]:
#Creating DataLoaders
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

print(len(train_loader))
print(len(test_loader))

30
4


In [25]:
#Loss + Optimizer
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    decoder.parameters(),
    lr=1e-3
)

In [26]:
#Training upto epoch 5
num_epochs = 30

for epoch in range(num_epochs):

    decoder.train()

    total_loss = 0

    for latent, target in train_loader:

        latent = latent.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        logits = decoder(latent)

        loss = criterion(
            logits.view(-1, 21),
            target.view(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {avg_loss:.4f}"
    )

Epoch [1/30] Loss: 1.9032
Epoch [2/30] Loss: 1.4503
Epoch [3/30] Loss: 1.4459
Epoch [4/30] Loss: 1.4443
Epoch [5/30] Loss: 1.4417
Epoch [6/30] Loss: 1.4444
Epoch [7/30] Loss: 1.4416
Epoch [8/30] Loss: 1.4438
Epoch [9/30] Loss: 1.4410
Epoch [10/30] Loss: 1.4427
Epoch [11/30] Loss: 1.4420
Epoch [12/30] Loss: 1.4383
Epoch [13/30] Loss: 1.4394
Epoch [14/30] Loss: 1.4390
Epoch [15/30] Loss: 1.4358
Epoch [16/30] Loss: 1.4373
Epoch [17/30] Loss: 1.4345
Epoch [18/30] Loss: 1.4357
Epoch [19/30] Loss: 1.4330
Epoch [20/30] Loss: 1.4330
Epoch [21/30] Loss: 1.4336
Epoch [22/30] Loss: 1.4311
Epoch [23/30] Loss: 1.4286
Epoch [24/30] Loss: 1.4262
Epoch [25/30] Loss: 1.4274
Epoch [26/30] Loss: 1.4262
Epoch [27/30] Loss: 1.4241
Epoch [28/30] Loss: 1.4245
Epoch [29/30] Loss: 1.4189
Epoch [30/30] Loss: 1.4205


In [27]:
decoder.eval()

correct = 0
total = 0

with torch.no_grad():

    for latent, target in test_loader:

        latent = latent.to(device)
        target = target.to(device)

        logits = decoder(latent)

        pred = logits.argmax(dim=2)

        correct += (
            pred == target
        ).sum().item()

        total += target.numel()

accuracy = correct / total

print(
    f"Token Accuracy: {accuracy:.4f}"
)

Token Accuracy: 0.6223
